# Notebook contains code for training Vision Transformer for prediction of violent activity

## Importing required libraries:

In [1]:
import os
import cv2
import torch
import random
import shutil
from tqdm import tqdm
from PIL import Image
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import random_split
import torchvision.transforms as transforms
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from transformers import ViTFeatureExtractor, ViTForImageClassification, CLIPProcessor, CLIPModel

## Extracting frames:

In [2]:
# Define target path to dataset
TARGET_PATH = "/kaggle/input/real-time-anomaly-detection-in-cctv-surveillance/data"
SELECTED_CATEGORIES = ["fighting", "abuse", "arson", "burglary", "shooting", "vandalism"]
EXTRACTED_FRAMES_PATH = "./extracted_frames"

# Step 1: Extract frames from videos
print("Step 1: Extracting frames from videos...")
os.makedirs(EXTRACTED_FRAMES_PATH, exist_ok=True)

for category in SELECTED_CATEGORIES:
    category_path = os.path.join(TARGET_PATH, category)
    save_path = os.path.join(EXTRACTED_FRAMES_PATH, category)
    os.makedirs(save_path, exist_ok=True)
    
    if os.path.exists(category_path):
        videos = [os.path.join(category_path, vid) for vid in os.listdir(category_path) if vid.endswith(".mp4")]
        for vid_path in videos:
            cap = cv2.VideoCapture(vid_path)
            frame_count = 0
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break
                if frame_count % 10 == 0:  # Save every 10th frame
                    frame_filename = f"{os.path.basename(vid_path).split('.')[0]}_frame{frame_count}.jpg"
                    frame_filepath = os.path.join(save_path, frame_filename)
                    cv2.imwrite(frame_filepath, frame)
                frame_count += 1
            cap.release()
        print(f"Extracted frames from {category}")
    else:
        print(f"Warning: Category folder not found - {category}")

print("Frame extraction completed!")

Step 1: Extracting frames from videos...
Extracted frames from fighting
Extracted frames from abuse
Extracted frames from arson
Extracted frames from burglary
Extracted frames from shooting
Extracted frames from vandalism
Frame extraction completed!


## Custom dataset class:

In [3]:
# Step 2: Define custom dataset class
class CCTVAnomalyDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        
        for idx, category in enumerate(SELECTED_CATEGORIES):
            category_path = os.path.join(root_dir, category)
            if os.path.exists(category_path):
                images = [os.path.join(category_path, img) for img in os.listdir(category_path) if img.endswith(".jpg")]
                self.image_paths.extend(images)
                self.labels.extend([idx] * len(images))
        
        if len(self.image_paths) == 0:
            raise ValueError("No images found in the dataset!")
        print(f"Dataset initialized with {len(self.image_paths)} images.")
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

## Applying necessary transformations:

In [4]:
# Step 3: Apply transformations and create DataLoader
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

print("Step 3: Loading dataset...")
dataset = CCTVAnomalyDataset(EXTRACTED_FRAMES_PATH, transform=transform)

# Define dataset sizes
total_size = len(dataset)
train_size = int(0.75 * total_size)  # 70% for training
val_size = int(0.10 * total_size)   # 10% for validation
test_size = total_size - train_size - val_size  # 15% for testing

# Split dataset
train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

print(f"Dataset split: {train_size} train, {val_size} val, {test_size} test samples")


batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print("DataLoaders created successfully!")

Step 3: Loading dataset...
Dataset initialized with 99501 images.
Dataset split: 74625 train, 9950 val, 14926 test samples
DataLoaders created successfully!


## Fine-tuning Vision Transformer:

In [5]:
# Step 4: Fine-Tuning ViT
print("Initializing ViT model...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=len(SELECTED_CATEGORIES),  # Ensure this matches your dataset
    ignore_mismatched_sizes=True
).to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=2e-5)
scaler = GradScaler()  # For mixed-precision training

# Training & validation function
def train_and_validate(model, train_loader, val_loader, epochs=5):
    """Train the model on train_loader and validate on val_loader."""
    model.train()

    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        
        # Training phase
        model.train()
        train_loss, correct, total = 0, 0, 0
        for images, labels in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()

            with autocast():  # Use FP16 for reduced memory usage
                outputs = model(images).logits
                loss = criterion(outputs, labels)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        train_accuracy = 100 * correct / total
        print(f"Train Loss: {train_loss/len(train_loader):.4f}, Train Accuracy: {train_accuracy:.2f}%")
        
        # Validation phase
        model.eval()
        val_loss, correct, total = 0, 0, 0
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc=f"Validating Epoch {epoch+1}"):
                images, labels = images.to(device), labels.to(device)
                outputs = model(images).logits
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        val_accuracy = 100 * correct / total
        print(f"Validation Loss: {val_loss/len(val_loader):.4f}, Validation Accuracy: {val_accuracy:.2f}%")

    print("Training & validation completed!")

# **Train and Validate**
train_and_validate(model, train_loader, val_loader, epochs=5)

Initializing ViT model...


config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([6]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([6, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-5-3cb68cb7b6f9>:13: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()  # For mixed-precision training



Epoch 1/5


...

Validation Loss: 0.0062, Validation Accuracy: 99.80%

Epoch 5/5


Training Epoch 5: 100%|██████████| 2333/2333 [21:44<00:00,  1.79it/s]


Train Loss: 0.0052, Train Accuracy: 99.81%


Validating Epoch 5: 100%|██████████| 311/311 [00:58<00:00,  5.32it/s]

Validation Loss: 0.0062, Validation Accuracy: 99.70%
Training & validation completed!


In [6]:
# Testing function
def test_model(model, test_loader):
    """Evaluate the model on the test set."""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Testing Model"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images).logits
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    test_accuracy = 100 * correct / total
    print(f"\nTest Accuracy: {test_accuracy:.2f}%")

test_model(model, test_loader)

Testing Model: 100%|██████████| 467/467 [01:26<00:00,  5.38it/s]


Test Accuracy: 99.71%


## Saving trained model:

In [18]:
print(model.state_dict().keys())  # Print model parameters before saving

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
odict_keys(['vit.embeddings.cls_token', 'vit.embeddings.position_embeddings', 'vit.embeddings.patch_embeddings.projection.weight', 'vit.embeddings.patch_embeddings.projection.bias', 'vit.encoder.layer.0.attention.attention.query.weight', 'vit.encoder.layer.0.attention.attention.query.bias', 'vit.encoder.layer.0.attention.attention.key.weight', 'vit.encoder.layer.0.attention.attention.key.bias', 'vit.encoder.layer.0.attention.attention.value.weight', 'vit.encoder.layer.0.attention.attention.value.bias', 'vit.encoder.layer.0.attention.output.dense.weight', 'vit.encoder.layer.0.attention.output.dense.bias', 'vit.encoder.layer.0.intermediate.dense.weight', 'vit.encoder.layer.0.intermediate.dense.bias', 'vit.encoder.layer.0.output.dense.weight', 'vit.encoder.layer.0.output.dense.bias', 'vit.encoder.layer.0.layernorm_before.weight', 'vit.en

In [7]:
# Step 5: Save the trained model
MODEL_SAVE_PATH = "/kaggle/working/vit_anomaly_detector.pth"
torch.save(model.state_dict(), MODEL_SAVE_PATH)
print(f"Model saved at {MODEL_SAVE_PATH}")

Model saved at /kaggle/working/vit_anomaly_detector.pth


## Loading CLIP for multi-modal approach (not needed):

In [8]:
# # Step 6: Load CLIP for Multi-Modal Recognition
# print("Loading CLIP model...")
# clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16").to(device)
# clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")

Loading CLIP model...


config.json:   0%|          | 0.00/4.10k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/599M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/961k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [16]:
# # Step 7: Run inference using ViT + CLIP
# # Define activity descriptions
# activity_descriptions = {
#     'burglary': 'A person breaking into a building or entering from a window or a chimney.',
#     'arson': 'A person setting fire to something.',
#     'shooting': 'A person pointing or shooting/firing a pistol/Piston or a weapon/Weapon.',
#     'abuse': 'A person physically harming another person.',
#     'vandalism': 'A person damaging property or breaking stuff.',
#     'normal': 'No suspicious or violent activity.'
# }
# SELECTED_CATEGORIES = list(activity_descriptions.keys())

# # Image transformation for ViT
# transform = transforms.Compose([
#     transforms.Resize((224, 224)),
#     transforms.ToTensor(),
# ])

# def predict_activity(frame):
#     image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    
#     # ViT Prediction
#     image_tensor = transform(image).unsqueeze(0).to(device)
#     model.eval()
#     with torch.no_grad():
#         vit_outputs = model(image_tensor).logits
#         vit_predicted_class = vit_outputs.argmax(dim=1).item()
#     vit_prediction = SELECTED_CATEGORIES[vit_predicted_class]
    
#     # CLIP Prediction
#     image_inputs = clip_processor(images=image, return_tensors="pt").to(device)
#     text_inputs = clip_processor(text=[activity_descriptions[a] for a in SELECTED_CATEGORIES], 
#                                  return_tensors="pt", padding=True).to(device)
    
#     with torch.no_grad():
#         clip_outputs = clip_model(**image_inputs, **text_inputs)
#         logits_per_image = clip_outputs.logits_per_image
#         probs = logits_per_image.softmax(dim=-1).cpu().numpy()
#     clip_prediction = SELECTED_CATEGORIES[probs.argmax()]
    
#     return vit_prediction, clip_prediction

# def process_video(input_video, output_video):
#     cap = cv2.VideoCapture(input_video)
#     width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
#     height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
#     fps = int(cap.get(cv2.CAP_PROP_FPS))
#     fourcc = cv2.VideoWriter_fourcc(*'mp4v')
#     out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))
    
#     while cap.isOpened():
#         ret, frame = cap.read()
#         if not ret:
#             break
        
#         vit_pred, clip_pred = predict_activity(frame)
        
#         # Draw prediction on frame
#         text = f"ViT: {vit_pred} | CLIP: {clip_pred}"
#         cv2.putText(frame, text, (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
#         out.write(frame)
    
#     cap.release()
#     out.release()

# # Example usage
# process_video("/kaggle/input/real-time-anomaly-detection-in-cctv-surveillance/data/fighting/Fighting002_x264.mp4", "/kaggle/working/fighting_output.mp4")
# print("Video saved.")

Video saved.


## Prediction on custom videos:

In [17]:
# Step 6: Video frame prediction with ViT
def predict_video(video_path, model):
    cap = cv2.VideoCapture(video_path)
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    out = cv2.VideoWriter('output.avi', fourcc, 30.0, (int(cap.get(3)), int(cap.get(4))))
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)).convert("RGB")
        image_tensor = transform(image).unsqueeze(0).to(device)
        
        model.eval()
        with torch.no_grad():
            outputs = model(image_tensor).logits
            predicted_class = outputs.argmax(dim=1).item()
        predicted_label = SELECTED_CATEGORIES[predicted_class]
        
        cv2.putText(frame, predicted_label, (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        out.write(frame)
    
    cap.release()
    out.release()
    print("Video processing completed!")

# Example usage
video_path = "/kaggle/input/real-time-anomaly-detection-in-cctv-surveillance/data/fighting/Fighting002_x264.mp4"
predict_video(video_path, model)

Video processing completed!
